# Cell 1 — Install dependencies on Kaggle
Run this on Kaggle with GPU T4 x2 enabled.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "tensorflow==2.17.0", "numpy==1.26.4", "pandas==2.2.2",
    "scikit-learn==1.3.2", "scipy==1.11.4", "matplotlib==3.8.3",
    "seaborn==0.13.0", "nltk==3.8.1", "transformers==4.35.2",
    "torch==2.1.2", "plotly==5.18.0", "protobuf==4.24.4"
], capture_output=True)

import tensorflow as tf
import numpy as np
import sklearn
import transformers
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"transformers: {transformers.__version__}")

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("Kaggle environment is STABLE — NO CONFLICTS!")

# Cell 2 — Data Loading
Merge GoEmotions + EmpatheticDialogues + ISEAR into one CSV.
Add all three datasets via 'Add Data' before running.

In [ ]:
import pandas as pd
import os

# These datasets should be added to your Kaggle notebook via "Add Data"
# Required: GoEmotions, EmpatheticDialogues, ISEAR Dataset
# Map them all to 5 classes: Bored, Confident, Confused, Curious, Frustrated

TARGET_EMOTIONS = ['Bored', 'Confident', 'Confused', 'Curious', 'Frustrated']

# --- Load GoEmotions (adjust path based on Kaggle dataset mount) ---
# GoEmotions has 27 emotion labels; map to our 5
go_emotions_map = {
    'boredom': 'Bored', 'nervousness': 'Bored',
    'admiration': 'Confident', 'approval': 'Confident', 'pride': 'Confident',
    'confusion': 'Confused', 'embarrassment': 'Confused',
    'curiosity': 'Curious', 'surprise': 'Curious',
    'anger': 'Frustrated', 'annoyance': 'Frustrated', 'disappointment': 'Frustrated',
    'disgust': 'Frustrated', 'grief': 'Frustrated', 'remorse': 'Frustrated'
}

# Load from Kaggle input path (update paths as needed)
dfs = []

# GoEmotions
try:
    go_path = '/kaggle/input/goemotions/go_emotions_dataset.csv'
    go_df = pd.read_csv(go_path)
    # GoEmotions format has 'text' and emotion columns (one-hot)
    # Melt and map to our 5 classes
    emotion_cols = [c for c in go_df.columns if c in go_emotions_map]
    for col in emotion_cols:
        subset = go_df[go_df[col] == 1][['text']].copy()
        subset['emotion'] = go_emotions_map[col]
        dfs.append(subset)
    print(f"GoEmotions loaded")
except Exception as e:
    print(f"GoEmotions skipped: {e}")

# EmpatheticDialogues
try:
    emp_map = {
        'bored': 'Bored', 'content': 'Confident', 'proud': 'Confident',
        'confused': 'Confused', 'curious': 'Curious', 'surprised': 'Curious',
        'angry': 'Frustrated', 'frustrated': 'Frustrated', 'anxious': 'Frustrated'
    }
    emp_path = '/kaggle/input/empatheticdialogues/train.csv'
    emp_df = pd.read_csv(emp_path)
    emp_df = emp_df[['utterance', 'context']].rename(columns={'utterance': 'text', 'context': 'emotion_raw'})
    emp_df['emotion'] = emp_df['emotion_raw'].str.lower().map(emp_map)
    emp_df = emp_df.dropna(subset=['emotion'])[['text', 'emotion']]
    dfs.append(emp_df)
    print(f"EmpatheticDialogues loaded")
except Exception as e:
    print(f"EmpatheticDialogues skipped: {e}")

# ISEAR
try:
    isear_map = {
        'joy': 'Confident', 'fear': 'Frustrated', 'anger': 'Frustrated',
        'sadness': 'Bored', 'disgust': 'Frustrated', 'shame': 'Confused', 'guilt': 'Confused'
    }
    isear_path = '/kaggle/input/isear-dataset/isear.csv'
    isear_df = pd.read_csv(isear_path)
    isear_df.columns = [c.lower() for c in isear_df.columns]
    isear_df['emotion'] = isear_df['emotion'].str.lower().map(isear_map)
    isear_df = isear_df.dropna(subset=['emotion'])[['text', 'emotion']]
    dfs.append(isear_df)
    print(f"ISEAR loaded")
except Exception as e:
    print(f"ISEAR skipped: {e}")

# Combine
combined_df = pd.concat(dfs, ignore_index=True)
combined_df = combined_df[combined_df['emotion'].isin(TARGET_EMOTIONS)]
combined_df = combined_df.dropna(subset=['text', 'emotion'])
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total samples: {len(combined_df)}")
print(combined_df['emotion'].value_counts())
combined_df.to_csv('/kaggle/working/emotion_text_dataset.csv', index=False)

# Cell 3 — Text Preprocessing & Tokenization

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

STOPWORDS = set(stopwords.words('english'))
MAX_VOCAB_SIZE = 30000
MAX_SEQ_LEN = 80

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

print("Cleaning text...")
combined_df['clean_text'] = combined_df['text'].apply(clean_text)

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(combined_df['clean_text'])
sequences = tokenizer.texts_to_sequences(combined_df['clean_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

print(f"Tokenization complete: {padded_sequences.shape}")
print(f"Classes: {TARGET_EMOTIONS}")

import pickle
with open('/kaggle/working/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
np.save('/kaggle/working/padded_sequences.npy', padded_sequences)
combined_df.to_csv('/kaggle/working/combined_preprocessed.csv', index=False)

# Cell 4 — BiLSTM Model Training
Architecture: Embedding(128) → BiLSTM(128 units) → Dense(5 softmax)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(combined_df['emotion'])
label_classes = label_encoder.classes_
print(f"Label classes: {label_classes}")
np.save('/kaggle/working/label_classes.npy', label_classes)

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Focal Loss for class imbalance
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        y_true_one_hot = tf.one_hot(y_true, depth=len(TARGET_EMOTIONS))
        ce = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1)
        pt = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, gamma) * ce
        return tf.reduce_mean(focal)
    return loss_fn

# BiLSTM Architecture: Embedding(128) → BiLSTM(128 units) → Dense(5 softmax)
model = Sequential([
    Embedding(MAX_VOCAB_SIZE, 128, input_length=MAX_SEQ_LEN),
    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(TARGET_EMOTIONS), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss=focal_loss(gamma=2.0),
    metrics=['accuracy']
)
model.summary()

early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping]
)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

# Save baseline
model.save('/kaggle/working/bilstm_baseline.keras')

# Cell 5 — Domain-Adaptive Fine-Tuning on Student Data
Generates 10,000 synthetic student-specific samples then fine-tunes at lr=1e-4.

In [ ]:
# Create 10,000 synthetic student-specific samples for fine-tuning
student_samples = [
    ("I don't understand recursion at all, how does the stack work?", "Confused"),
    ("I finally get dynamic programming! Memoization makes sense now", "Confident"),
    ("Integration by parts is so boring, when will I ever use this", "Bored"),
    ("Why does gradient descent converge? I'm curious about the math", "Curious"),
    ("I've tried this problem 10 times and I still can't get it", "Frustrated"),
    # ... generate more programmatically
]

# Generate 2000 samples per class synthetically
templates = {
    'Bored': ["This topic is so {} and {}", "I find {} really {} and unengaging"],
    'Confident': ["I understand {} perfectly now", "I've mastered {} and feel great about it"],
    'Confused': ["I don't get {} at all", "Can someone explain {} to me?"],
    'Curious': ["I wonder how {} works exactly", "I'm fascinated by {} and want to learn more"],
    'Frustrated': ["I can't figure out {} no matter what", "This {} is impossible and I'm giving up"]
}

topics = ["recursion", "calculus", "neural networks", "linear algebra", "sorting algorithms",
          "probability", "thermodynamics", "chemical bonds", "photosynthesis", "supply curves"]

import random
random.seed(42)

student_texts, student_labels = [], []
for emotion, tmpl_list in templates.items():
    for _ in range(2000):
        tmpl = random.choice(tmpl_list)
        topic1 = random.choice(topics)
        topic2 = random.choice(topics)
        try:
            text = tmpl.format(topic1, topic2)
        except Exception:
            text = tmpl.format(topic1)
        student_texts.append(text)
        student_labels.append(emotion)

student_df = pd.DataFrame({'text': student_texts, 'emotion': student_labels})
student_df['clean_text'] = student_df['text'].apply(clean_text)

student_seqs = tokenizer.texts_to_sequences(student_df['clean_text'])
student_padded = pad_sequences(student_seqs, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
student_y = label_encoder.transform(student_df['emotion'])

# Fine-tune with lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=focal_loss(2.0), metrics=['accuracy'])
model.fit(student_padded, student_y, epochs=5, batch_size=32, validation_split=0.1)

val_loss, val_acc = model.evaluate(student_padded, student_y)
print(f"Student Adaptive Validation Accuracy: {val_acc:.4f}")

model.save('/kaggle/working/bilstm_student_adaptive.keras')
print("BiLSTM student-adaptive model saved!")

# Cell 6 — BERT Fine-Tuning
Fine-tune bert-base-uncased on 40k samples for 3 epochs.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import Dataset, DataLoader
import torch
import json

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            truncation=True, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Use a subset for BERT (it's slower) — 40k samples
bert_df = combined_df.sample(n=min(40000, len(combined_df)), random_state=42)
bert_texts = bert_df['clean_text'].tolist()
bert_labels = label_encoder.transform(bert_df['emotion']).tolist()

X_tr, X_te, y_tr, y_te = train_test_split(bert_texts, bert_labels, test_size=0.1, random_state=42)
train_dataset = EmotionDataset(X_tr, y_tr, bert_tokenizer)
test_dataset = EmotionDataset(X_te, y_te, bert_tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5).to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

# Train 3 epochs
for epoch in range(3):
    bert_model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
bert_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
print(f"BERT Test Accuracy: {correct/total:.4f}")

# Save full HuggingFace suite
bert_model.save_pretrained('/kaggle/working/bert_emotion_model_final')
bert_tokenizer.save_pretrained('/kaggle/working/bert_emotion_model_final')

# Save label mapping
label_mapping = {str(i): cls for i, cls in enumerate(label_classes)}
with open('/kaggle/working/bert_emotion_model_final/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

print("BERT model + tokenizer + label_mapping.json saved!")

# Cell 7 — Verify all output files exist before downloading

In [ ]:
import os

required_files = [
    '/kaggle/working/bilstm_student_adaptive.keras',
    '/kaggle/working/tokenizer.pkl',
    '/kaggle/working/label_classes.npy',
    '/kaggle/working/bert_emotion_model_final/config.json',
    '/kaggle/working/bert_emotion_model_final/model.safetensors',
    '/kaggle/working/bert_emotion_model_final/tokenizer.json',
    '/kaggle/working/bert_emotion_model_final/label_mapping.json',
    '/kaggle/working/emotion_text_dataset.csv',
]

print("Verifying output files...")
all_good = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"{status} {f}")
    if not exists:
        all_good = False

if all_good:
    print("\n🎉 All files present! Download and place them in your local project.")
    print("\nDownload instructions:")
    print("1. bilstm_student_adaptive.keras → models/bltsm/")
    print("2. tokenizer.pkl → models/bltsm/")
    print("3. label_classes.npy → models/bltsm/")
    print("4. bert_emotion_model_final/ (entire folder) → models/")
    print("5. emotion_text_dataset.csv → data/")
else:
    print("\n⚠️ Some files missing. Check the cells above for errors.")